# TSC GRPO 标准化训练

简化版的交通信号配时优化 GRPO 训练流程：
1. **阶段一**：通过仿真生成 Dataset（遍历所有配时组合，计算 rewards）
2. **阶段二**：使用 Unsloth GRPOTrainer 标准流程进行模型微调

## 特点
- 最小绿/最大绿在合理范围内随机生成
- 遍历所有相位持续时间组合（步长5秒）
- 使用仿真状态保存/恢复实现反事实评估
- 统计 training step 的 rewards 和 KL divergence

## 1. 环境配置

In [ ]:
import os
import sys
import json
import random
import gc
from typing import Dict, List, Tuple, Any, Optional
from dataclasses import dataclass
from collections import deque
import xml.etree.ElementTree as ET

# 设置环境变量
os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"

# 网络代理配置（如需要）
import subprocess
result = subprocess.run('bash -c "source /etc/network_turbo && env | grep proxy"', shell=True, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if '=' in line:
        var, value = line.split('=', 1)
        os.environ[var] = value

print("环境配置完成")

## 2. 场景发现 + 仿真器初始化

In [ ]:
# 添加仿真模块路径
sumo_sim_path = os.path.join(os.getcwd(), 'sumo_simulation')
if sumo_sim_path not in sys.path:
    sys.path.insert(0, sumo_sim_path)

from sumo_simulator import SUMOSimulator


def extract_traffic_light_ids(net_xml_path: str) -> List[str]:
    """从 net.xml 提取信号灯 ID"""
    tl_ids: List[str] = []
    try:
        for _event, elem in ET.iterparse(net_xml_path, events=("end",)):
            if elem.tag == "tlLogic":
                tl_id = elem.attrib.get("id")
                if tl_id and tl_id not in tl_ids:
                    tl_ids.append(tl_id)
                elem.clear()
    except Exception as e:
        print(f"解析 {net_xml_path} 失败: {e}")
    return tl_ids


def discover_environments(environments_root: str, max_scenarios: int = 50) -> Dict[str, Dict]:
    """
    发现可用的仿真场景
    
    Args:
        environments_root: environments 目录路径
        max_scenarios: 最大场景数（避免加载过多）
    """
    environments: Dict[str, Dict] = {}
    if not os.path.isdir(environments_root):
        print(f"警告: environments 目录不存在: {environments_root}")
        return environments

    scenario_names = sorted(os.listdir(environments_root))
    random.shuffle(scenario_names)  # 随机顺序
    
    count = 0
    for scenario_name in scenario_names:
        if count >= max_scenarios:
            break
            
        scenario_dir = os.path.join(environments_root, scenario_name)
        if not os.path.isdir(scenario_dir) or scenario_name.startswith('.'):
            continue

        # 找 .sumocfg 文件
        sumocfg = None
        for f in os.listdir(scenario_dir):
            if f.endswith('.sumocfg'):
                sumocfg = os.path.join(scenario_dir, f)
                break

        # 找 .net.xml 文件
        net_xml = None
        for f in os.listdir(scenario_dir):
            if f.endswith('.net.xml'):
                net_xml = os.path.join(scenario_dir, f)
                break

        if sumocfg and net_xml:
            tl_ids = extract_traffic_light_ids(net_xml)
            if tl_ids:
                environments[scenario_name] = {
                    'sumocfg': sumocfg,
                    'net': net_xml,
                    'tl_ids': tl_ids[:50],  # 每个场景最多5个信号灯
                }
                count += 1

    return environments


# 发现场景
ENVIRONMENTS_ROOT = os.path.join(os.getcwd(), 'sumo_simulation/environments')
AVAILABLE_ENVIRONMENTS = discover_environments(ENVIRONMENTS_ROOT, max_scenarios=3000)

print(f"发现场景数: {len(AVAILABLE_ENVIRONMENTS)}")
total_intersections = sum(len(env['tl_ids']) for env in AVAILABLE_ENVIRONMENTS.values())
print(f"总交叉口数量: {total_intersections}")

# 显示前几个场景
for name, info in list(AVAILABLE_ENVIRONMENTS.items())[:3]:
    print(f"  - {name}: {len(info['tl_ids'])} 个信号灯")

## 3. Dataset 生成函数（核心）

### 核心逻辑：
1. 遍历每个场景的每个信号灯
2. 随机生成 min_green / max_green（合理范围）
3. 保存仿真状态
4. 遍历所有相位持续时间组合（步长5秒）
5. 执行仿真，计算 reward
6. 随机选择一个配时方案推进仿真

In [ ]:
# ==================== 配置参数 ====================
DATASET_CONFIG = {
    'duration_step': 5,           # 相位持续时间步长（秒）
    'min_green_range': (10, 30),  # 最小绿灯时间范围
    'max_green_range': (40, 90),  # 最大绿灯时间范围
    'warmup_steps': 300,          # 预热步数
    'cycle_duration': 60,         # 每个配时方案执行的仿真时长（秒）
    'gui': True,                 # 是否显示 GUI（生成数据时关闭以加速）
    'samples_per_tl': 10,         # 每个信号灯生成的样本数
}

# ==================== Prompt 模板 ====================
SYSTEM_PROMPT = """你是交通信号配时优化专家。
你将收到一个 JSON（用【cycle_predict_input_json】...【/cycle_predict_input_json】包裹）。
你的任务是：在满足硬约束前提下，输出下一周期各相位最终绿灯时间 final（单位：秒）。
注意：history 数据可能不完整，必须基于可用部分输出结果。
只输出最终 JSON 数组(list)，不要输出任何解释/过程。
"""


def sample_phase_limits(phase_order: List[int], rng: random.Random = None) -> Dict[str, Dict[str, int]]:
    """
    为每个相位随机生成 min_green / max_green
    
    Args:
        phase_order: 相位顺序列表 [1, 2, 3, 4]
        rng: 随机数生成器
    
    Returns:
        {"1": {"min_green": 15, "max_green": 60}, ...}
    """
    if rng is None:
        rng = random.Random()
    
    min_lo, min_hi = DATASET_CONFIG['min_green_range']
    max_lo, max_hi = DATASET_CONFIG['max_green_range']
    
    limits = {}
    for phase_id in phase_order:
        min_green = rng.randint(min_lo, min_hi)
        max_green = rng.randint(max_lo, max_hi)
        # 确保 max > min
        if max_green <= min_green:
            max_green = min_green + 20
        limits[str(phase_id)] = {"min_green": min_green, "max_green": max_green}
    
    return limits


def generate_all_timing_combinations(phase_limits: Dict[str, Dict[str, int]], 
                                      step: int = 5) -> List[List[Dict[str, int]]]:
    """
    生成所有相位持续时间的组合
    
    Args:
        phase_limits: {"1": {"min_green": 15, "max_green": 60}, ...}
        step: 步长（秒）
    
    Returns:
        List of timing plans, each plan is [{"phase_id": 1, "final": 20}, ...]
    """
    from itertools import product
    
    phase_ids = sorted([int(k) for k in phase_limits.keys()])
    
    # 为每个相位生成可能的持续时间
    phase_durations = []
    for pid in phase_ids:
        lim = phase_limits[str(pid)]
        mn, mx = lim["min_green"], lim["max_green"]
        durations = list(range(mn, mx + 1, step))
        if not durations:
            durations = [mn]
        phase_durations.append(durations)
    
    # 生成所有组合
    all_combinations = []
    for combo in product(*phase_durations):
        plan = [{"phase_id": pid, "final": dur} for pid, dur in zip(phase_ids, combo)]
        all_combinations.append(plan)
    
    return all_combinations


def build_prompt(scenario_name: str, tl_id: str, phase_order: List[int],
                 phase_limits: Dict[str, Dict[str, int]], 
                 recent_waits: List[Dict] = None) -> str:
    """
    构建训练用的 prompt
    """
    import datetime
    import hashlib
    
    # 生成 crossing_id
    h = hashlib.md5(f"{scenario_name}::{tl_id}".encode("utf-8")).hexdigest()
    crossing_id = int(h[:8], 16)
    
    # 构建 payload
    payload = {
        "crossing_id": crossing_id,
        "as_of": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "phase_order": phase_order,
        "phase_limits": phase_limits,
        "history": {
            "recent_cycles": recent_waits or [],
            "cycle_times": [],
            "windows": {
                "recent_past": None,
                "yesterday_same_time": None,
                "last_week_same_time": None,
            },
        },
    }
    
    json_text = json.dumps(payload, ensure_ascii=False, separators=(",", ":"))
    prompt_content = f"【cycle_predict_input_json】{json_text}【/cycle_predict_input_json】"
    
    return SYSTEM_PROMPT + "\n\n" + prompt_content


def plan_to_completion(plan: List[Dict[str, int]]) -> str:
    """将配时方案转换为 completion 字符串"""
    return json.dumps(plan, ensure_ascii=False, separators=(",", ":"))


print("Dataset 生成函数定义完成")

In [ ]:
def evaluate_plan_in_simulation(simulator: SUMOSimulator, tl_id: str, 
                                 plan: List[Dict[str, int]], 
                                 duration: int = 60) -> Dict[str, float]:
    """
    在仿真中评估一个配时方案
    
    Args:
        simulator: SUMO 仿真器
        tl_id: 信号灯 ID
        plan: 配时方案 [{"phase_id": 1, "final": 30}, ...]
        duration: 执行时长（秒）
    
    Returns:
        {"passed_vehicles": ..., "queue_vehicles": ..., "total_waiting_time": ...}
    """
    import traci
    
    # 获取所有受控车道
    controlled_lanes = list(set(traci.trafficlight.getControlledLanes(tl_id)))
    
    # 记录开始时的车辆
    vehicles_before = set()
    for lane in controlled_lanes:
        try:
            vehicles_before.update(traci.lane.getLastStepVehicleIDs(lane))
        except:
            pass
    
    # 执行配时方案
    total_waiting_time = 0.0
    for phase_item in plan:
        phase_idx = phase_item["phase_id"] - 1  # 转为 0-based
        phase_duration = phase_item["final"]
        
        try:
            traci.trafficlight.setPhase(tl_id, phase_idx)
        except:
            continue
        
        # 执行该相位的持续时间
        for _ in range(phase_duration):
            try:
                traci.simulationStep()
                # 累计等待时间
                for lane in controlled_lanes:
                    try:
                        total_waiting_time += traci.lane.getWaitingTime(lane)
                    except:
                        pass
            except:
                break
    
    # 记录结束时的车辆
    vehicles_after = set()
    queue_vehicles = 0
    for lane in controlled_lanes:
        try:
            vehicles_after.update(traci.lane.getLastStepVehicleIDs(lane))
            queue_vehicles += traci.lane.getLastStepHaltingNumber(lane)
        except:
            pass
    
    # 通过车辆数 = 开始时存在但结束时不存在的车辆
    passed_vehicles = len(vehicles_before - vehicles_after)
    
    return {
        "passed_vehicles": passed_vehicles,
        "queue_vehicles": queue_vehicles,
        "total_waiting_time": total_waiting_time,
    }


def compute_reward(result: Dict[str, float], 
                   alpha: float = 1.0, 
                   beta: float = 0.5, 
                   gamma: float = 0.1) -> float:
    """
    计算 reward
    
    Args:
        result: 仿真结果 {"passed_vehicles": ..., "queue_vehicles": ..., "total_waiting_time": ...}
        alpha: 通过车辆权重
        beta: 排队车辆惩罚权重
        gamma: 等待时间惩罚权重
    
    Returns:
        reward in [-1, 1] (normalized)
    """
    import math
    
    passed = result.get("passed_vehicles", 0)
    queue = result.get("queue_vehicles", 0)
    wait = result.get("total_waiting_time", 0)
    
    # 原始 reward
    raw_reward = alpha * passed - beta * queue - gamma * wait / 100
    
    # 使用 tanh 归一化到 [-1, 1]
    normalized = math.tanh(raw_reward / 20.0)
    
    return normalized


def collect_phase_waits(simulator: SUMOSimulator, tl_id: str, 
                        phase_order: List[int]) -> List[Dict]:
    """收集当前各相位的等待车辆数作为 history"""
    import traci
    
    waits = []
    for phase_id in phase_order:
        phase_idx = phase_id - 1
        lanes_info = simulator.get_phase_controlled_lanes(tl_id, phase_idx)
        lanes = lanes_info.get('incoming_lanes', []) if lanes_info else []
        
        if not lanes:
            avg = 0.0
        else:
            total = 0.0
            for ln in lanes:
                try:
                    total += traci.lane.getLastStepHaltingNumber(ln)
                except:
                    pass
            avg = total / max(1, len(lanes))
        
        waits.append({'phase_id': int(phase_id), 'avg_wait': float(round(avg, 2))})
    
    return waits


print("仿真评估函数定义完成")

In [ ]:
def generate_dataset_for_scenario(scenario_name: str, env_info: Dict,
                                   samples_per_tl: int = 10,
                                   max_combos_per_sample: int = 50) -> List[Dict]:
    """
    为单个场景生成 Dataset
    
    Args:
        scenario_name: 场景名称
        env_info: 场景信息 {"sumocfg": ..., "net": ..., "tl_ids": [...]}
        samples_per_tl: 每个信号灯生成的样本数
        max_combos_per_sample: 每个样本评估的最大组合数
    
    Returns:
        List of dataset samples
    """
    import traci
    
    dataset = []
    rng = random.Random()
    
    # 初始化仿真器
    simulator = SUMOSimulator(
        config_file=env_info['sumocfg'],
        junctions_file=None,
        gui=DATASET_CONFIG['gui'],
    )
    
    try:
        if not simulator.start_simulation():
            print(f"  ✗ 场景 {scenario_name} 启动失败")
            return dataset
        
        print(f"  ✓ 场景 {scenario_name} 已启动")
        
        # 遍历每个信号灯
        for tl_id in env_info['tl_ids']:
            print(f"    处理信号灯: {tl_id}")
            
            # 获取相位信息
            phase_info = simulator.get_phase_info(tl_id)
            num_phases = phase_info.get('num_phases', 0)
            if num_phases == 0:
                print(f"      ✗ 无相位信息，跳过")
                continue
            
            phase_order = list(range(1, num_phases + 1))
            
            # 为每个信号灯生成多个样本
            for sample_idx in range(samples_per_tl):
                try:
                    # 1. 随机生成 min_green / max_green
                    phase_limits = sample_phase_limits(phase_order, rng)
                    
                    # 2. 生成所有配时组合
                    all_combos = generate_all_timing_combinations(
                        phase_limits, 
                        step=DATASET_CONFIG['duration_step']
                    )
                    
                    # 限制组合数量
                    if len(all_combos) > max_combos_per_sample:
                        all_combos = rng.sample(all_combos, max_combos_per_sample)
                    
                    if not all_combos:
                        continue
                    
                    # 3. 收集当前交通状态作为 history
                    recent_waits = collect_phase_waits(simulator, tl_id, phase_order)
                    
                    # 4. 保存仿真状态
                    state_file = f"temp_state_{scenario_name}_{tl_id}_{sample_idx}.xml"
                    traci.simulation.saveState(state_file)
                    
                    # 5. 遍历所有组合，评估并保存
                    combo_rewards = []
                    for plan in all_combos:
                        # 恢复状态
                        traci.simulation.loadState(state_file)
                        
                        # 评估该配时方案
                        result = evaluate_plan_in_simulation(simulator, tl_id, plan)
                        reward = compute_reward(result)
                        combo_rewards.append((plan, reward, result))
                    
                    # 6. 构建 prompt
                    prompt = build_prompt(
                        scenario_name, tl_id, phase_order, phase_limits,
                        [{"time": "now", "phase_waits": recent_waits}]
                    )
                    
                    # 7. 为每个组合生成一个样本
                    for plan, reward, result in combo_rewards:
                        completion = plan_to_completion(plan)
                        dataset.append({
                            "prompt": prompt,
                            "completion": completion,
                            "reward": reward,
                            "metadata": {
                                "scenario": scenario_name,
                                "tl_id": tl_id,
                                "passed": result["passed_vehicles"],
                                "queue": result["queue_vehicles"],
                            }
                        })
                    
                    # 8. 随机选择一个方案推进仿真
                    traci.simulation.loadState(state_file)
                    chosen_plan = rng.choice(all_combos)
                    evaluate_plan_in_simulation(simulator, tl_id, chosen_plan)
                    
                    # 清理状态文件
                    try:
                        os.remove(state_file)
                    except:
                        pass
                    
                except Exception as e:
                    print(f"      ✗ 样本 {sample_idx} 生成失败: {e}")
                    continue
            
            print(f"      ✓ 生成 {len([d for d in dataset if d.get('metadata', {}).get('tl_id') == tl_id])} 个样本")
        
    except Exception as e:
        print(f"  ✗ 场景 {scenario_name} 处理失败: {e}")
    finally:
        simulator.close()
    
    return dataset


print("Dataset 生成器定义完成")

## 4. 执行 Dataset 生成

生成约 20000+ 样本用于 GRPO 训练。

In [ ]:
# ==================== 生成 Dataset ====================
DATASET_OUTPUT_FILE = "tsc_grpo_dataset.json"

def generate_full_dataset(target_samples: int = 20000) -> List[Dict]:
    """
    生成完整的 Dataset
    
    Args:
        target_samples: 目标样本数
    
    Returns:
        完整的 dataset
    """
    all_dataset = []
    scenario_list = list(AVAILABLE_ENVIRONMENTS.items())
    
    print(f"开始生成 Dataset，目标: {target_samples} 样本")
    print(f"可用场景: {len(scenario_list)}")
    print("=" * 60)
    
    # 计算每个场景需要的样本数
    samples_per_scenario = max(1, target_samples // len(scenario_list))
    
    for idx, (scenario_name, env_info) in enumerate(scenario_list):
        print(f"\n[{idx+1}/{len(scenario_list)}] 处理场景: {scenario_name}")
        
        # 计算每个信号灯的样本数
        num_tls = len(env_info['tl_ids'])
        samples_per_tl = max(1, samples_per_scenario // num_tls)
        
        # 生成该场景的数据
        scenario_dataset = generate_dataset_for_scenario(
            scenario_name, env_info,
            samples_per_tl=samples_per_tl,
            max_combos_per_sample=50
        )
        
        all_dataset.extend(scenario_dataset)
        print(f"  累计样本数: {len(all_dataset)}")
        
        # 如果已达到目标，可以提前停止
        if len(all_dataset) >= target_samples:
            print(f"\n已达到目标样本数 {target_samples}，停止生成")
            break
        
        # 定期保存
        if (idx + 1) % 5 == 0:
            print(f"\n  [保存中间结果: {len(all_dataset)} 样本]")
            with open(DATASET_OUTPUT_FILE, 'w', encoding='utf-8') as f:
                json.dump(all_dataset, f, ensure_ascii=False, indent=2)
    
    return all_dataset


# 执行生成（可以调整目标样本数）
# 注意：这个过程可能需要较长时间，可以先用小样本测试
print("=" * 60)
print("Dataset 生成器已准备就绪")
print("执行 generate_full_dataset() 开始生成")
print("=" * 60)

In [ ]:
# 执行 Dataset 生成
# 可以调整 target_samples 参数控制生成数量

dataset = generate_full_dataset(target_samples=200)

# 保存最终结果
with open(DATASET_OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

print(f"\n" + "=" * 60)
print(f"Dataset 生成完成!")
print(f"总样本数: {len(dataset)}")
print(f"保存到: {DATASET_OUTPUT_FILE}")
print("=" * 60)

# 显示样本统计
if dataset:
    rewards = [d['reward'] for d in dataset]
    print(f"\nReward 统计:")
    print(f"  - 平均值: {sum(rewards)/len(rewards):.4f}")
    print(f"  - 最大值: {max(rewards):.4f}")
    print(f"  - 最小值: {min(rewards):.4f}")
    
    # 显示一个样本示例
    print(f"\n样本示例:")
    sample = dataset[0]
    print(f"  Prompt (前200字符): {sample['prompt'][:200]}...")
    print(f"  Completion: {sample['completion']}")
    print(f"  Reward: {sample['reward']:.4f}")

---

# 阶段二：Unsloth GRPO 标准训练

## 5. 加载模型 + PEFT 配置

In [ ]:
from unsloth import FastLanguageModel
import torch

# ==================== 模型配置 ====================
MODEL_CONFIG = {
    'model_name': "model/models/qwen3-4B-SFT",  # 基础模型路径
    'max_seq_length': 2048,
    'lora_rank': 32,
    'load_in_4bit': False,
}

# 设置环境变量
os.environ["HF_HOME"] = 'model'
os.environ["MODELSCOPE_CACHE"] = 'model'

# 加载模型
print("加载预训练模型...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_CONFIG['model_name'],
    max_seq_length=MODEL_CONFIG['max_seq_length'],
    load_in_4bit=MODEL_CONFIG['load_in_4bit'],
    fast_inference=False,
    max_lora_rank=MODEL_CONFIG['lora_rank'],
    gpu_memory_utilization=0.8,
)

# 配置 PEFT (LoRA)
print("配置 LoRA...")
model = FastLanguageModel.get_peft_model(
    model,
    r=MODEL_CONFIG['lora_rank'],
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=MODEL_CONFIG['lora_rank'] * 2,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("✓ 模型加载完成")

## 6. 加载 Dataset + 定义 Reward 函数

In [ ]:
from datasets import load_dataset, Dataset
import re

# ==================== 加载 Dataset ====================
print("加载 Dataset...")

# 从 JSON 文件加载
raw_data = json.load(open(DATASET_OUTPUT_FILE, 'r', encoding='utf-8'))

# 转换为 Hugging Face Dataset 格式
# GRPO 需要的格式: {"prompt": str} 
# completions 和 rewards 会在训练时通过 reward_func 计算
hf_dataset = Dataset.from_list([{"prompt": d["prompt"]} for d in raw_data])

print(f"✓ 加载了 {len(hf_dataset)} 个样本")
print(f"  样本示例 prompt (前100字符): {hf_dataset[0]['prompt'][:100]}...")


# ==================== 定义 Reward 函数 ====================
def parse_completion_json(text: str) -> Optional[List[Dict]]:
    """解析 completion 中的 JSON 数组"""
    if not text:
        return None
    s = text.strip()
    # 查找 JSON 数组
    m = re.search(r"\[[\s\S]*\]", s)
    if not m:
        return None
    try:
        obj = json.loads(m.group(0))
        if not isinstance(obj, list):
            return None
        # 验证格式
        for it in obj:
            if not isinstance(it, dict):
                return None
            if "phase_id" not in it or "final" not in it:
                return None
        return obj
    except:
        return None


def extract_phase_limits_from_prompt(prompt: str) -> Optional[Dict]:
    """从 prompt 中提取 phase_limits"""
    try:
        # 查找 JSON 部分
        start = prompt.find("【cycle_predict_input_json】")
        end = prompt.find("【/cycle_predict_input_json】")
        if start == -1 or end == -1:
            return None
        json_str = prompt[start + len("【cycle_predict_input_json】"):end]
        payload = json.loads(json_str)
        return payload.get("phase_limits")
    except:
        return None


def tsc_reward_func(completions: List[str], prompts: List[str], **kwargs) -> List[float]:
    """
    TSC 任务的 Reward 函数
    
    评估标准：
    1. 格式正确性 (+0.3)
    2. 满足约束 (+0.4)
    3. 配时合理性 (+0.3)
    """
    rewards = []
    
    for completion, prompt in zip(completions, prompts):
        reward = 0.0
        
        # 1. 解析 completion
        plan = parse_completion_json(completion)
        if plan is None:
            rewards.append(-1.0)  # 格式错误，强惩罚
            continue
        
        reward += 0.3  # 格式正确
        
        # 2. 检查约束
        phase_limits = extract_phase_limits_from_prompt(prompt)
        if phase_limits:
            bounds_ok = True
            for item in plan:
                pid = str(item.get("phase_id", ""))
                final = item.get("final", 0)
                if pid in phase_limits:
                    mn = phase_limits[pid].get("min_green", 0)
                    mx = phase_limits[pid].get("max_green", 999)
                    if final < mn or final > mx:
                        bounds_ok = False
                        break
            
            if bounds_ok:
                reward += 0.4  # 满足约束
            else:
                reward -= 0.2  # 违反约束
        
        # 3. 配时合理性（简单启发式：总时长在合理范围内）
        total_duration = sum(item.get("final", 0) for item in plan)
        if 60 <= total_duration <= 300:
            reward += 0.3
        elif 30 <= total_duration <= 400:
            reward += 0.1
        else:
            reward -= 0.1
        
        rewards.append(reward)
    
    return rewards


print("✓ Reward 函数定义完成")

## 7. GRPOConfig + GRPOTrainer

In [ ]:
from trl import GRPOTrainer, GRPOConfig

# ==================== GRPO 配置 ====================
GRPO_OUTPUT_DIR = "grpo_tsc_output"

grpo_config = GRPOConfig(
    # 输出目录
    output_dir=GRPO_OUTPUT_DIR,
    
    # 训练参数
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    
    # GRPO 特定参数
    num_generations=4,           # 每个 prompt 生成的 completion 数量
    temperature=0.8,             # 生成温度（增加多样性）
    top_p=0.95,                  # Top-p 采样
    top_k=50,                    # Top-k 采样
    max_new_tokens=256,          # 最大生成长度
    
    # KL 散度控制
    kl_coef=0.1,                 # KL 惩罚系数
    
    # 优化器参数
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    
    # 日志和保存
    logging_steps=10,
    save_steps=100,
    save_total_limit=3,
    
    # 其他
    bf16=True,                   # 使用 BF16 精度
    seed=3407,
    
    # 报告
    report_to="none",            # 可以改为 "wandb" 使用 W&B
)

print("✓ GRPO 配置完成")
print(f"  - num_generations: {grpo_config.num_generations}")
print(f"  - temperature: {grpo_config.temperature}")
print(f"  - kl_coef: {grpo_config.kl_coef}")
print(f"  - learning_rate: {grpo_config.learning_rate}")

## 8. 训练 + 统计可视化

In [ ]:
# ==================== 训练统计回调 ====================
from transformers import TrainerCallback

class GRPOStatsCallback(TrainerCallback):
    """
    自定义回调，用于记录训练过程中的 rewards 和 KL divergence
    """
    def __init__(self):
        self.step_rewards = []      # 每步的平均 reward
        self.step_kl_divs = []      # 每步的 KL divergence
        self.step_losses = []       # 每步的 loss
        self.current_step = 0
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        """每次 log 时记录统计信息"""
        if logs is None:
            return
        
        self.current_step = state.global_step
        
        # 记录 reward
        if 'reward' in logs:
            self.step_rewards.append({
                'step': self.current_step,
                'reward': logs['reward']
            })
        elif 'rewards/mean' in logs:
            self.step_rewards.append({
                'step': self.current_step,
                'reward': logs['rewards/mean']
            })
        
        # 记录 KL divergence
        if 'kl' in logs:
            self.step_kl_divs.append({
                'step': self.current_step,
                'kl': logs['kl']
            })
        elif 'kl_divergence' in logs:
            self.step_kl_divs.append({
                'step': self.current_step,
                'kl': logs['kl_divergence']
            })
        
        # 记录 loss
        if 'loss' in logs:
            self.step_losses.append({
                'step': self.current_step,
                'loss': logs['loss']
            })
    
    def get_stats(self):
        """获取统计信息"""
        return {
            'rewards': self.step_rewards,
            'kl_divergence': self.step_kl_divs,
            'losses': self.step_losses,
        }


# 创建回调实例
stats_callback = GRPOStatsCallback()

print("✓ 统计回调已创建")

In [ ]:
# ==================== 创建 Trainer 并开始训练 ====================

print("创建 GRPOTrainer...")

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=hf_dataset,
    reward_funcs=[tsc_reward_func],
    args=grpo_config,
    callbacks=[stats_callback],
)

print("✓ Trainer 已创建")
print("=" * 60)
print("开始 GRPO 训练...")
print("=" * 60)

# 开始训练
train_result = trainer.train()

print("\n" + "=" * 60)
print("训练完成!")
print("=" * 60)

In [ ]:
# ==================== 统计可视化 ====================
import matplotlib.pyplot as plt

def plot_training_stats(stats: Dict):
    """绘制训练统计图"""
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # 1. Rewards 曲线
    ax1 = axes[0]
    if stats['rewards']:
        steps = [r['step'] for r in stats['rewards']]
        rewards = [r['reward'] for r in stats['rewards']]
        ax1.plot(steps, rewards, 'b-', linewidth=1.5)
        ax1.set_xlabel('Training Step')
        ax1.set_ylabel('Mean Reward')
        ax1.set_title('Training Rewards')
        ax1.grid(True, alpha=0.3)
    else:
        ax1.text(0.5, 0.5, 'No reward data', ha='center', va='center')
        ax1.set_title('Training Rewards')
    
    # 2. KL Divergence 曲线
    ax2 = axes[1]
    if stats['kl_divergence']:
        steps = [r['step'] for r in stats['kl_divergence']]
        kl_divs = [r['kl'] for r in stats['kl_divergence']]
        ax2.plot(steps, kl_divs, 'r-', linewidth=1.5)
        ax2.set_xlabel('Training Step')
        ax2.set_ylabel('KL Divergence')
        ax2.set_title('KL Divergence')
        ax2.grid(True, alpha=0.3)
    else:
        ax2.text(0.5, 0.5, 'No KL data', ha='center', va='center')
        ax2.set_title('KL Divergence')
    
    # 3. Loss 曲线
    ax3 = axes[2]
    if stats['losses']:
        steps = [r['step'] for r in stats['losses']]
        losses = [r['loss'] for r in stats['losses']]
        ax3.plot(steps, losses, 'g-', linewidth=1.5)
        ax3.set_xlabel('Training Step')
        ax3.set_ylabel('Loss')
        ax3.set_title('Training Loss')
        ax3.grid(True, alpha=0.3)
    else:
        ax3.text(0.5, 0.5, 'No loss data', ha='center', va='center')
        ax3.set_title('Training Loss')
    
    plt.tight_layout()
    plt.savefig('grpo_training_stats.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ 统计图已保存到 grpo_training_stats.png")


# 获取并显示统计信息
stats = stats_callback.get_stats()

print("\n" + "=" * 60)
print("训练统计")
print("=" * 60)

if stats['rewards']:
    reward_values = [r['reward'] for r in stats['rewards']]
    print(f"Rewards:")
    print(f"  - 数据点数: {len(reward_values)}")
    print(f"  - 平均值: {sum(reward_values)/len(reward_values):.4f}")
    print(f"  - 最大值: {max(reward_values):.4f}")
    print(f"  - 最小值: {min(reward_values):.4f}")
    print(f"  - 最终值: {reward_values[-1]:.4f}")

if stats['kl_divergence']:
    kl_values = [r['kl'] for r in stats['kl_divergence']]
    print(f"\nKL Divergence:")
    print(f"  - 数据点数: {len(kl_values)}")
    print(f"  - 平均值: {sum(kl_values)/len(kl_values):.4f}")
    print(f"  - 最大值: {max(kl_values):.4f}")
    print(f"  - 最终值: {kl_values[-1]:.4f}")

if stats['losses']:
    loss_values = [r['loss'] for r in stats['losses']]
    print(f"\nLoss:")
    print(f"  - 数据点数: {len(loss_values)}")
    print(f"  - 初始值: {loss_values[0]:.4f}")
    print(f"  - 最终值: {loss_values[-1]:.4f}")

# 绘制图表
plot_training_stats(stats)

# 保存统计数据
with open('grpo_training_stats.json', 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)
print("✓ 统计数据已保存到 grpo_training_stats.json")

## 9. 保存模型

In [ ]:
# ==================== 保存模型 ====================
SAVE_MODEL_DIR = "grpo_tsc_model"

print("保存微调后的模型...")

# 保存 LoRA 适配器
model.save_pretrained(SAVE_MODEL_DIR)
tokenizer.save_pretrained(SAVE_MODEL_DIR)

print(f"✓ LoRA 适配器已保存到: {SAVE_MODEL_DIR}")

# 可选：合并并保存完整模型
SAVE_MERGED_DIR = "grpo_tsc_model_merged"

print("\n合并 LoRA 到基础模型...")
merged_model = model.merge_and_unload()
merged_model.save_pretrained(SAVE_MERGED_DIR)
tokenizer.save_pretrained(SAVE_MERGED_DIR)

print(f"✓ 合并后的完整模型已保存到: {SAVE_MERGED_DIR}")

print("\n" + "=" * 60)
print("所有任务完成!")
print("=" * 60)
print(f"1. Dataset: {DATASET_OUTPUT_FILE}")
print(f"2. LoRA 模型: {SAVE_MODEL_DIR}/")
print(f"3. 合并模型: {SAVE_MERGED_DIR}/")
print(f"4. 训练统计: grpo_training_stats.json")
print(f"5. 统计图表: grpo_training_stats.png")